<a id="understanding-quickstart"></a>
# VideoDB Understanding Quickstart

Create two reusable, timestamped artifacts: a speech transcript and a visual scene description. No index is created yet.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a id="install"></a>
## 1. Install and connect

In [ ]:
!pip install -q "videodb>=0.5.0" python-dotenv pandas

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()
print("Connected to VideoDB")
print("Collection:", collection.id)

<a id="video"></a>
## 2. Choose a video

The default clip contains dialogue and clear scene changes. Use the commented `VIDEO_ID` lines to reuse a user-specific upload, or set `VIDEODB_VIDEO_URL` to test another public video.

In [ ]:
VIDEO_URL = os.getenv(
    "VIDEODB_VIDEO_URL",
    "https://www.youtube.com/watch?v=vVlEVRKv4is",  # Silicon Valley - Gilfoyle is free for hire
)

collection = conn.get_collection()
video = collection.upload(url=VIDEO_URL)

# To reuse a video already uploaded to your account, comment out the upload above
# and use your own VideoDB video ID:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)
video.play()


<a id="run"></a>
## 3. Create an Understanding run

Each analyzer produces one named **artifact**. Assign explicit names so later cells and pipeline inputs remain stable:

- `spoken_words`, named `transcript` (what was said)
- `vlm` (vision-language model), named `scene` (what is visible or happening)

They run independently in this first example. The pipeline guide shows how to pass transcript or OCR artifacts into a downstream VLM.

> Go deeper: [segmentation and sampling](segmentation-and-sampling.ipynb) · [multi-analyzer pipelines](multi-analyzer-pipelines.ipynb)

In [ ]:
understanding = video.understand(
    analyzers=[
        {"type": "spoken_words", "name": "transcript"},
        {
            "type": "vlm",
            "name": "scene",
            "sampling": {"strategy": "uniform", "frame_count": 3},
            "config": {
                "model": "pro",
                "prompt": (
                    "Describe what visibly happens in this scene. Mention the main people, "
                    "objects, actions, and setting. Do not infer details that are not visible."
                ),
            },
        },
    ],
)

print("Understanding created")
print(f"ID: {understanding.id}")
print(f"Status: {understanding.status}")

<a id="wait"></a>
## 4. Wait for completion

An Understanding can contain parallel and dependent analyzers. Wait for the run, then inspect each analyzer.

In [ ]:
understanding.wait_until_complete(timeout=3600, poll_interval=15)
print(f"Final status: {understanding.status}")

print("\nAnalyzer statuses:")
for analyzer in understanding.list_analyzers():
    print(f"- {analyzer.name} ({analyzer.type}): {analyzer.status}")

<a id="outputs"></a>
## 5. Fetch and display outputs

Analyzer output uses a common scene envelope: `scene_id`, `start`, `end`, and analyzer-specific `data`. The transcript stores `data.text` and timed words; the VLM stores its free-form description in `data.text`.

> For statuses, callbacks, existing runs, and failures, see [Outputs and operations](outputs-and-operations.ipynb).

In [ ]:
import pandas as pd
from IPython.display import display

transcript_output = understanding.get_analyzer("transcript").get_output()
scene_output = understanding.get_analyzer("scene").get_output()

transcript_scenes = transcript_output.get("scenes", [])
transcript_rows = [
    {
        "start": scene.get("start"),
        "end": scene.get("end"),
        "text": (scene.get("data") or {}).get("text"),
        "language": (scene.get("data") or {}).get("language"),
        "word_count": len((scene.get("data") or {}).get("words", [])),
    }
    for scene in transcript_scenes
]

scene_scenes = scene_output.get("scenes", [])
scene_rows = [
    {"start": scene.get("start"), "end": scene.get("end"), **(scene.get("data") or {})}
    for scene in scene_scenes
]

print(f"Transcript ({len(transcript_rows)} scenes)")
display(pd.DataFrame(transcript_rows).head())

print(f"\nScene understanding ({len(scene_rows)} scenes)")
display(pd.DataFrame(scene_rows).head())

<a id="existing"></a>
## 6. Resume an existing run

In [ ]:
same_understanding = video.get_understanding(understanding.id)
print("Fetched Understanding")
print(f"ID: {same_understanding.id}")
print(f"Status: {same_understanding.status}")

print("\nAvailable Understandings:")
for item in video.list_understandings():
    print(f"- {item.id}: {item.status}")

<a id="next"></a>
## Next steps

| Goal | Notebook |
|---|---|
| Explore every analyzer | [Understanding guide map](README.md) |
| Tune scenes and frames | [Segmentation and sampling](segmentation-and-sampling.ipynb) |
| Compose dependent analyzers | [Multi-analyzer pipelines](multi-analyzer-pipelines.ipynb) |
| Use managed VLM models and schemas | [VLM with Managed Models](vlm/managed-models.ipynb) |
| Run a self-hosted VLM | [VLM with Sandbox Models](vlm/sandbox-models.ipynb) |
| Index these outputs | [Indexing guide](../indexing/indexing_guide.ipynb) |

<a id="cleanup"></a>
## Optional cleanup

In [ ]:
DELETE_UNDERSTANDING = False

if DELETE_UNDERSTANDING:
    understanding.delete()
    print("Deleted", understanding.id)
else:
    print("Skipping delete")